In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
from pathlib import Path

import pandas as pd

from datasmith import setup_environment
from datasmith.docker.context import ContextRegistry

setup_environment()

/mnt/sdd1/atharvas/formulacode/datasmith


09:16:49 WARNING  simple_useragent.core: Falling back to historic user agent.


In [2]:
cmd = """
python scratch/scripts/synthesize_contexts.py
    --commits {output_dir}/commits_perfonly.parquet
    --output-dir {output_dir}/results_synthesis/
    --context-registry {output_dir}/context_registry.json
    --max-workers {ncpus}
    --limit-per-repo 2
    --max-attempts 3
    --max-steps 10
""".strip().replace("\n", " ")

In [3]:
cr = ContextRegistry.load_from_file(Path("scratch/artifacts/pipeflush/context_registry.json"))
commit_pth = Path("scratch/artifacts/pipeflush/commits_perfonly.parquet")
commit_df = pd.read_parquet(commit_pth)
commit_df.head()

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,patch,has_asv,file_change_summary,kind,repo_name
0,3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8,2024-07-06T09:38:32+08:00,Merge pull request #125 from Kai-Striega/broad...,133,66,3,numpy_financial/_financial.py\nnumpy_financial...,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
1,3f67c275e1e575c902027ca07586b9d35f38033a,2024-05-07T15:04:23+10:00,Merge pull request #122 from Eugenia-Mazur/irr...,62,47,1,numpy_financial/_financial.py,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
2,5c66fb06ec95192d4b427b4de171b6ab9e1528a6,2024-05-04T11:03:28+10:00,Merge pull request #124 from Kai-Striega/confi...,8,18,3,asv.conf.json\ndoc/source/dev/running_the_benc...,From 646f292a26089dc212e4315f0939c183f660ccea ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
3,6c40b8efb727eacf8a865789afbe65ee2d4bb5c0,2024-04-04T14:13:19+11:00,Merge pull request #120 from Kai-Striega/enh/n...,6,2,1,numpy_financial/_cfinancial.pyx,From 5b134ac31419fea11db1dda25315d1bd192d8430 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
4,858358697fce8fb96530f9c299d285286e5192e5,2024-04-04T10:36:54+11:00,Merge pull request #118 from Kai-Striega/enh/n...,95,29,3,numpy_financial/_cfinancial.pyx\nnumpy_financi...,From 6b6f7b5ba1a50a1199c408b99538c397ef54d0ba ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial


In [4]:
print(commit_df[["repo_name", "sha"]])

                   repo_name                                       sha
0      numpy/numpy-financial  3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8
1      numpy/numpy-financial  3f67c275e1e575c902027ca07586b9d35f38033a
2      numpy/numpy-financial  5c66fb06ec95192d4b427b4de171b6ab9e1528a6
3      numpy/numpy-financial  6c40b8efb727eacf8a865789afbe65ee2d4bb5c0
4      numpy/numpy-financial  858358697fce8fb96530f9c299d285286e5192e5
...                      ...                                       ...
46947        PostHog/posthog  f079a1efe07a3dd062967ba4bc72078177149953
46948        PostHog/posthog  f832d5783e20180261f490e04d40d329af819bdd
46949        PostHog/posthog  f9dd7ef5d6823cb9a6dbf1e91fe51fb3873375ee
46950        PostHog/posthog  fe12a8b9945b8a6c14252003ad983327c662fe7b
46951        PostHog/posthog  fe82d214b66041c0a6e6026351a93da52e153163

[46952 rows x 2 columns]


In [16]:
# # Break parquet file into 6 chunks. Try to put equal number of repos in each chunk.
# n_chunks = 6
# # (1036, 7)
# chunk_size = commit_df.shape[0] // n_chunks
# # shuffle rows and split into chunks
# commit_df = commit_df.sample(frac=1, random_state=42, replace=False).reset_index(drop=True)
# chunks = [commit_df.iloc[i * chunk_size : (i + 1) * chunk_size] for i in range(n_chunks - 1)]
# chunks.append(commit_df.iloc[(n_chunks - 1) * chunk_size :])  # last chunk gets the remainder
# cmds = []
# for i, chunk in enumerate(chunks):
#     pth = Path(f"scratch/artifacts/pipeflush/chunk_{i}/commits_perfonly.parquet")
#     pth.parent.mkdir(parents=True, exist_ok=True)
#     chunk.to_parquet(pth)
#     # Make a new context registry:
#     cr.save_to_file(pth.parent / "context_registry.json")
#     cmd_i = cmd.format(output_dir=pth.parent)
#     cmds.append(cmd_i)

In [5]:
# break parquet into three chunks with fixed ratios.
ratios = [64, 56, 127]
total = sum(ratios)
# Compute split sizes
sizes = [int(commit_df.shape[0] * r / total) for r in ratios]

# Adjust last size to cover remainder (to avoid row loss due to rounding)
sizes[-1] = commit_df.shape[0] - sum(sizes[:-1])

# Split dataframe
df1 = commit_df.iloc[: sizes[0]]
df2 = commit_df.iloc[sizes[0] : sizes[0] + sizes[1]]
df3 = commit_df.iloc[sizes[0] + sizes[1] :]

chunks = [df1, df2, df3]
cmds = []
for i, (chunk, ratio) in enumerate(zip(chunks, ratios)):
    pth = Path(f"scratch/artifacts/pipeflush/chunk_{i}/commits_perfonly.parquet")
    pth.parent.mkdir(parents=True, exist_ok=True)
    chunk.to_parquet(pth)
    # Make a new context registry:
    cr.save_to_file(pth.parent / "context_registry.json")
    cmd_i = cmd.format(output_dir=pth.parent, ncpus=(ratio // 2))
    cmds.append(cmd_i)

09:17:16 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/pipeflush/chunk_0/context_registry.json
09:17:36 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/pipeflush/chunk_1/context_registry.json
09:17:58 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/pipeflush/chunk_2/context_registry.json


In [6]:
print("\n".join(cmds).replace("  ", ""))

python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_0/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_0/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_0/context_registry.json --max-workers 32 --limit-per-repo 2 --max-attempts 3 --max-steps 10
python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_1/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_1/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_1/context_registry.json --max-workers 28 --limit-per-repo 2 --max-attempts 3 --max-steps 10
python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_2/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_2/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_2/context_registry.json --max-workers 63 --limit-per-repo 2 --max-attempts 3 --max-steps 10


In [ ]:
# # make a tiny task with just two commits for testing
# tiny = commit_df.sample(n=2, random_state=141, replace=False).reset_index(drop=True)
# tiny_pth = Path("scratch/artifacts/pipeflush/tiny/commits_perfonly.parquet")
# tiny_pth.parent.mkdir(parents=True, exist_ok=True)
# # tiny.to_parquet(tiny_pth)
# cr.save_to_file(tiny_pth.parent / "context_registry.json")
# cmd_tiny = cmd.format(output_dir=tiny_pth.parent, ncpus=2)
# print(cmd_tiny)


07:25:05 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/pipeflush/tiny/context_registry.json


python scratch/scripts/synthesize_contexts.py     --commits scratch/artifacts/pipeflush/tiny/commits_perfonly.parquet     --output-dir scratch/artifacts/pipeflush/tiny/results_synthesis/     --context-registry scratch/artifacts/pipeflush/tiny/context_registry.json     --max-workers 2     --limit-per-repo 2     --max-attempts 3     --max-steps 10
